In [1]:
import json
import math
import pickle
import re
import unicodedata
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 13})

### Load Data

In [3]:
data_dir = Path('../../datasets/processed_data')
split_dir = Path('../../datasets/split_data')

# Processed BIO sequences
with open(data_dir / 'processed.pkl', 'rb') as f:
    processed = pickle.load(f)

# Vocabularies
with open(data_dir / 'vocabs.pkl', 'rb') as f:
    v = pickle.load(f)
    Entity_labels = v['Entity_labels']
    token2id = v['token2id']
    id2token = v['id2token']
    label2id = v['label2id']
    id2label = v['id2label']

# Train / val / test split indices (shared across all models)
with open(split_dir / 'split_indices.json') as f:
    split = json.load(f)
idx_train = split['idx_train']
idx_val = split['idx_val']
idx_test = split['idx_test']

print(f"Resumes loaded : {len(processed)}")
print(f"Entity labels  : {Entity_labels}")
print(f"Train / Val / Test : {len(idx_train)} / {len(idx_val)} / {len(idx_test)}")

Resumes loaded : 220
Entity labels  : ['Name', 'Designation', 'Companies worked at', 'Location', 'Email Address', 'College Name', 'Degree', 'Graduation Year', 'Skills', 'Years of Experience']
Train / Val / Test : 154 / 33 / 33


### BIO Sequence Extraction

In [6]:
def get_sequences(indices):
    """Return (token_sequences, tag_sequences) for the given resume indices."""
    tokens_list, tags_list = [], []
    for i in indices:
        bio = processed[i]['word_bio_norm']
        tokens_list.append([tok for tok, _ in bio])
        tags_list.append([tag for _, tag in bio])
    return tokens_list, tags_list

train_tokens, train_tags = get_sequences(idx_train)
val_tokens, val_tags = get_sequences(idx_val)
test_tokens, test_tags = get_sequences(idx_test)

all_train_tokens = train_tokens + val_tokens  # used for final model
all_train_tags = train_tags   + val_tags

print(f"Train seqs : {len(train_tokens)}")
print(f"Val seqs   : {len(val_tokens)}")
print(f"Test seqs  : {len(test_tokens)}")

Train seqs : 154
Val seqs   : 33
Test seqs  : 33


### Word Similarity Feature

In [7]:
def char_trigrams(word: str) -> set:
    """Return the set of character trigrams for a word."""
    w = f"  {word}  "   # boundary padding
    return {w[i:i+3] for i in range(len(w) - 2)}


def trigram_similarity(w1: str, w2: str) -> float:
    """Jaccard similarity between the trigram sets of two words."""
    t1, t2 = char_trigrams(w1), char_trigrams(w2)
    inter = len(t1 & t2)
    union = len(t1 | t2)
    return inter / union if union else 0.0


def build_similarity_index(
    vocab: list[str],
    top_k: int = 10,
    min_sim: float = 0.2
) -> dict[str, list[tuple[str, float]]]:
    """
    For every word in vocab, precompute the top-k most similar words
    (by trigram Jaccard) with similarity >= min_sim.

    Returns
    -------
    sim_index : dict  word -> [(similar_word, similarity), ...]
    """
    sim_index = {}
    vocab_list = list(vocab)
    n = len(vocab_list)
    for i, w1 in enumerate(vocab_list):
        sims = []
        for j, w2 in enumerate(vocab_list):
            if i == j:
                continue
            s = trigram_similarity(w1, w2)
            if s >= min_sim:
                sims.append((w2, s))
        sims.sort(key=lambda x: -x[1])
        sim_index[w1] = sims[:top_k]
    return sim_index


# Build vocabulary from training tokens
train_vocab = sorted({tok for seq in all_train_tokens for tok in seq})
print(f"Training vocabulary size: {len(train_vocab):,}")
print("Building similarity index (may take ~30 s for large vocab) ...")

sim_index = build_similarity_index(train_vocab, top_k=10, min_sim=0.2)
print("Done.")

# Quick demo
demo_word = "engineer"
print(f"\nTop similar words to '{demo_word}':")
for w, s in sim_index.get(demo_word, [])[:5]:
    print(f"  {w:<25}  sim={s:.3f}")

Training vocabulary size: 7,815
Building similarity index (may take ~30 s for large vocab) ...
Done.

Top similar words to 'engineer':
  enginner                   sim=0.538
  engineering                sim=0.533
  engine                     sim=0.500
  enginerre                  sim=0.400
  reengineere                sim=0.353
